# FakeBDTeen Feature Extractor (Notebook 1)
This notebook builds a cached feature store by extracting aligned audio/video embeddings for all videos. Run the cells top-to-bottom once on GPU.

In [ ]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from transformers import Wav2Vec2Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

audio_encoder = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-xls-r-300m')
audio_encoder.eval()
for p in audio_encoder.parameters():
    p.requires_grad = False
audio_encoder.to(device)

video_encoder = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
video_encoder.fc = nn.Identity()
video_encoder.eval()
for p in video_encoder.parameters():
    p.requires_grad = False
video_encoder.to(device)

frame_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def compile_fakebdteen_registry(root_dir: str, output_csv: str) -> pd.DataFrame:
    records = []
    for gender in sorted(os.listdir(root_dir)):
        gender_path = os.path.join(root_dir, gender)
        if not os.path.isdir(gender_path):
            continue
        for authenticity_class in sorted(os.listdir(gender_path)):
            class_path = os.path.join(gender_path, authenticity_class)
            if not os.path.isdir(class_path):
                continue
            video_label = 1 if 'Fake_Video' in authenticity_class else 0
            audio_label = 1 if 'Fake_Audio' in authenticity_class else 0
            for language in sorted(os.listdir(class_path)):
                language_path = os.path.join(class_path, language)
                if not os.path.isdir(language_path):
                    continue
                for subject_id in sorted(os.listdir(language_path)):
                    subject_path = os.path.join(language_path, subject_id)
                    if not os.path.isdir(subject_path):
                        continue
                    for file_name in sorted(os.listdir(subject_path)):
                        if not file_name.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                            continue
                        try:
                            video_path = os.path.join(subject_path, file_name)
                            cap = cv2.VideoCapture(video_path)
                            if not cap.isOpened() or cap.get(cv2.CAP_PROP_FRAME_COUNT) <= 0:
                                cap.release()
                                print(f'Skipping invalid video: {file_name}')
                                continue
                            cap.release()
                        except Exception as e:
                            print(f'Error checking {file_name}: {e}')
                            continue
                        unique_id = f"{gender}_{authenticity_class}_{language}_{subject_id}_{os.path.splitext(file_name)[0]}"
                        records.append({
                            'unique_id': unique_id,
                            'gender': gender,
                            'authenticity_class': authenticity_class,
                            'language': language,
                            'subject_id': subject_id,
                            'video_path': video_path,
                            'video_label': video_label,
                            'audio_label': audio_label
                        })
    df = pd.DataFrame.from_records(records)
    df.to_csv(output_csv, index=False)
    return df

dataset_root = '/kaggle/input/datasets/tanmoykdas/fakebdteen/FakeBDTeen'
metadata_csv = '/kaggle/working/fakebdteen_metadata.csv'
registry_df = compile_fakebdteen_registry(dataset_root, metadata_csv)
print(registry_df.head())
print(f'Total videos: {len(registry_df)}')

## 1. Dependencies & Encoder Setup
Load frozen XLS-R and ResNet-18 backbones, configure device, and define frame transforms.

In [ ]:
def extract_audio_features(video_path: str, target_timesteps: int = 100) -> np.ndarray:
    waveform, sample_rate = torchaudio.load(video_path)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
    waveform = waveform.to(device)
    with torch.no_grad():
        audio_out = audio_encoder(waveform).last_hidden_state
        audio_out = audio_out.transpose(1, 2)
        audio_out = F.interpolate(audio_out, size=target_timesteps, mode='linear', align_corners=False)
        audio_out = audio_out.transpose(1, 2).squeeze(0)
    return audio_out.cpu().numpy().astype(np.float32)

def extract_video_features(video_path: str, target_timesteps: int = 100) -> np.ndarray:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Failed to open video: {video_path}')
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        raise RuntimeError(f'Empty video: {video_path}')
    indices = np.linspace(0, frame_count - 1, target_timesteps).astype(np.int64)
    frames = []
    current_idx = 0
    target_set = set(indices.tolist())
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if current_idx in target_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_transform(frame))
        current_idx += 1
    cap.release()
    if len(frames) != target_timesteps:
        if len(frames) == 0:
            raise RuntimeError(f'No frames sampled: {video_path}')
        last_frame = frames[-1]
        while len(frames) < target_timesteps:
            frames.append(last_frame)
        frames = frames[:target_timesteps]
    batch = torch.stack(frames, dim=0).to(device)
    with torch.no_grad():
        feats = video_encoder(batch)
    return feats.cpu().numpy().astype(np.float32)

## 2. Dataset Registry & Labels
Scan the dataset directory and create the metadata CSV with multi-task labels.

In [ ]:
from tqdm import tqdm

features_root = '/kaggle/working/features'
audio_root = os.path.join(features_root, 'audio')
video_root = os.path.join(features_root, 'video')
os.makedirs(audio_root, exist_ok=True)
os.makedirs(video_root, exist_ok=True)

error_count = 0
success_count = 0

for _, row in tqdm(registry_df.iterrows(), total=len(registry_df), desc='Extracting features'):
    unique_id = row['unique_id']
    video_path = row['video_path']
    audio_out_path = os.path.join(audio_root, f'{unique_id}.npy')
    video_out_path = os.path.join(video_root, f'{unique_id}.npy')
    if os.path.exists(audio_out_path) and os.path.exists(video_out_path):
        success_count += 1
        continue
    try:
        audio_features = extract_audio_features(video_path, target_timesteps=100)
        video_features = extract_video_features(video_path, target_timesteps=100)
        np.save(audio_out_path, audio_features, allow_pickle=False)
        np.save(video_out_path, video_features, allow_pickle=False)
        success_count += 1
    except Exception as e:
        error_count += 1
        print(f'Error processing {unique_id}: {str(e)}')

print(f'\nFeature Extraction Summary:')
print(f'Successfully extracted: {success_count}')
print(f'Errors: {error_count}')

## 3. Aligned Feature Extraction
Extract and temporally align audio/video features to 100 steps.

In [ ]:
audio_files = [f for f in os.listdir(audio_root) if f.endswith('.npy')]
video_files = [f for f in os.listdir(video_root) if f.endswith('.npy')]
print(f'Audio features: {len(audio_files)}')
print(f'Video features: {len(video_files)}')

if len(audio_files) > 0:
    sample_audio = np.load(os.path.join(audio_root, audio_files[0]))
    print(f'Sample audio shape: {sample_audio.shape}')
if len(video_files) > 0:
    sample_video = np.load(os.path.join(video_root, video_files[0]))
    print(f'Sample video shape: {sample_video.shape}')

def get_folder_size_mb(folder_path: str) -> float:
    total_bytes = 0
    for root, _, files in os.walk(folder_path):
        for name in files:
            fp = os.path.join(root, name)
            total_bytes += os.path.getsize(fp)
    return total_bytes / (1024 * 1024)

size_mb = get_folder_size_mb(features_root)
print(f'Feature store size: {size_mb:.2f} MB')

zip_path = '/kaggle/working/fakebdteen_extracted_features.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive('/kaggle/working/fakebdteen_extracted_features', 'zip', features_root)
print(f'Archive created at: {zip_path}')

## 4. Dataset Visualization & Feature QA
Plot dataset distribution and verify extracted feature quality.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# ---------- Dataset distribution ----------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('FakeBDTeen Dataset Overview', fontsize=16, fontweight='bold')

gender_counts = registry_df['gender'].value_counts().sort_index()
axes[0, 0].bar(gender_counts.index, gender_counts.values, color=['#4e79a7', '#f28e2b'])
axes[0, 0].set_title('Gender Distribution')
axes[0, 0].set_ylabel('Video Count')
for i, v in enumerate(gender_counts.values):
    axes[0, 0].text(i, v + max(5, int(0.01 * v)), str(v), ha='center', fontsize=10)

auth_counts = registry_df['authenticity_class'].value_counts()
axes[0, 1].barh(auth_counts.index, auth_counts.values, color='#59a14f')
axes[0, 1].set_title('Authenticity Class Distribution')
axes[0, 1].set_xlabel('Video Count')

lang_counts = registry_df['language'].value_counts().sort_index()
axes[1, 0].bar(lang_counts.index, lang_counts.values, color='#e15759')
axes[1, 0].set_title('Language Distribution')
axes[1, 0].set_ylabel('Video Count')
for i, v in enumerate(lang_counts.values):
    axes[1, 0].text(i, v + max(5, int(0.01 * v)), str(v), ha='center', fontsize=10)

combo_counts = registry_df.groupby(['video_label', 'audio_label']).size().reset_index(name='count')
combo_map = {
    (0, 0): 'Real Video + Real Audio',
    (1, 0): 'Fake Video + Real Audio',
    (0, 1): 'Real Video + Fake Audio',
    (1, 1): 'Fake Video + Fake Audio'
}
combo_counts['combo'] = combo_counts.apply(lambda r: combo_map[(int(r['video_label']), int(r['audio_label']))], axis=1)
combo_counts = combo_counts.sort_values('count', ascending=False)
axes[1, 1].barh(combo_counts['combo'], combo_counts['count'], color='#76b7b2')
axes[1, 1].set_title('Audio-Video Label Combination')
axes[1, 1].set_xlabel('Video Count')

plt.tight_layout()
plt.show()

# ---------- Extracted feature sanity check ----------
audio_files = sorted([f for f in os.listdir(audio_root) if f.endswith('.npy')])
video_files = sorted([f for f in os.listdir(video_root) if f.endswith('.npy')])

print('\n=== Feature Store Sanity Check ===')
print(f'Audio feature files: {len(audio_files)}')
print(f'Video feature files: {len(video_files)}')

if len(audio_files) > 0 and len(video_files) > 0:
    sample_n = min(20, len(audio_files), len(video_files))
    sample_audio_stats, sample_video_stats = [], []

    for fn in audio_files[:sample_n]:
        arr = np.load(os.path.join(audio_root, fn))
        sample_audio_stats.append([arr.mean(), arr.std(), arr.shape[0], arr.shape[1]])

    for fn in video_files[:sample_n]:
        arr = np.load(os.path.join(video_root, fn))
        sample_video_stats.append([arr.mean(), arr.std(), arr.shape[0], arr.shape[1]])

    sample_audio_stats = np.array(sample_audio_stats)
    sample_video_stats = np.array(sample_video_stats)

    print(f'Audio shape (expected ~100 x 1024): {tuple(np.load(os.path.join(audio_root, audio_files[0])).shape)}')
    print(f'Video shape (expected ~100 x 2048): {tuple(np.load(os.path.join(video_root, video_files[0])).shape)}')
    print(f'Audio mean/std across {sample_n} samples: {sample_audio_stats[:,0].mean():.4f} / {sample_audio_stats[:,1].mean():.4f}')
    print(f'Video mean/std across {sample_n} samples: {sample_video_stats[:,0].mean():.4f} / {sample_video_stats[:,1].mean():.4f}')

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(sample_audio_stats[:, 1], bins=10, color='#af7aa1', alpha=0.85)
    ax[0].set_title('Audio Feature Std Distribution')
    ax[0].set_xlabel('Std')

    ax[1].hist(sample_video_stats[:, 1], bins=10, color='#ff9da7', alpha=0.85)
    ax[1].set_title('Video Feature Std Distribution')
    ax[1].set_xlabel('Std')

    plt.tight_layout()
    plt.show()